# EPH Scrubbing × Semantic Bridge × Burst Analysis — Integrated Pipeline

An end-to-end Jupyter notebook combining three prior workflows:

1. **EPH boolean scrubbing** (EPHScrubbingv6) — researcher-defined keyword groups and boolean searches that partition a Reddit corpus into event-aligned sub-corpora (LA Fires, NC Helene floods, Texas floods) crossed with discourse flavors (Gov/Response, Disinformation, Misinformation).
2. **Load & Bridge** (semantic_bridge_pipeline) — LDA topic discovery, ETO Map-of-Science backbone mapping, LLM topic relabeling, and network visualization.
3. **Burst analysis** (new) — Kleinberg two-state burst detection on term frequency over time, aligned to each event's day-zero, so we can see how perceptions and framing shift across the arc of an event.

### Design choices that fix the "metadata over-emphasis" problem from prior runs

- Boolean searches are treated as the **top-level topics** — they are researcher-defined and grounded in decision theory. LDA is demoted to finding *sub-topics inside each boolean sub-corpus*.
- Content fields (`title`, `body`) and metadata fields (`subreddit`, `author`, `post_id`, `permalink`, `timestamp`) are separated at ingest. Only content goes to topic models.
- Platform-specific noise (`edit`, `deleted`, `removed`, `http`, `www`, etc.) is added to the stopword list automatically.


## 1. Setup

Paths, imports, and optional dependency installation. Run once per kernel session.


### 1.1 Configure paths

`DATA_DIR` should point to a folder of Reddit `.json` / `.jsonl` files. On TACC this is typically the Corral path shown below; locally it defaults to a `WORKING JSON` subfolder next to the notebook. `OUTPUT_DIR` receives all exports.


In [1]:
from pathlib import Path
import os

# ── Data source: pick whichever exists in your environment ──────────────
CANDIDATE_DATA_DIRS = [
    Path("/corral-repl/tacc/aci/PT2050/projects/PTDATAX-267/Data/For_HICCS"),
    Path("/corral-repl/tacc/aci/PT2050/projects/PTDATAX-267/Data/RedditScrubbingData"),
    Path("/corral-repl/tacc/aci/PT2050/projects/PTDATAX-267/Data/Reddit Scrubbing Data"),
    Path("C:/Users/ecm3479/OneDrive - The University of Texas at Austin/Documents/Media Scrubbing Worksheets/JSON"),
    Path(os.getcwd()) / "Subreddit_JSON"
]
DATA_DIR = next((p for p in CANDIDATE_DATA_DIRS if p.exists()), CANDIDATE_DATA_DIRS[-1])
OUTPUT_DIR = Path(os.getcwd()) / "IntegratedResults"
OUTPUT_DIR.mkdir(exist_ok=True)

# Event anchors for burst analysis (UTC date of major impact)
EVENT_DAY_ZERO = {
    "NC_Helene":  "2024-09-26",   # Helene landfall, western NC impacts follow
    "TX_Flood":   "2025-07-04",   # Kerr County / Guadalupe River flash flood
}
WINDOW_DAYS = 60  # ± days around day-zero for burst/temporal comparison

print(f"✓ DATA_DIR   = {DATA_DIR}   (exists: {DATA_DIR.exists()})")
print(f"✓ OUTPUT_DIR = {OUTPUT_DIR}")
print(f"✓ Events     = {list(EVENT_DAY_ZERO.keys())}")


✓ DATA_DIR   = C:\Users\ecm3479\OneDrive - The University of Texas at Austin\Documents\Media Scrubbing Worksheets\JSON   (exists: True)
✓ OUTPUT_DIR = c:\Users\ecm3479\OneDrive - The University of Texas at Austin\Documents\GitHub\mediascrubbing-floodhealth\IntegratedResults
✓ Events     = ['NC_Helene', 'TX_Flood']


### 1.2 Install and import dependencies

Everything uses mainstream packages available on TACC Jupyter kernels. `openpyxl` and `plotly` are installed on demand if missing. The Anthropic client is only needed if you enable LLM topic labeling in Section 4.


In [2]:
import sys, subprocess, importlib

def _ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for _p, _m in [("openpyxl", "openpyxl"), ("plotly", "plotly"),
               ("networkx", "networkx"), ("scikit-learn", "sklearn")]:
    _ensure(_p, _m)

import json, re, glob, math
from datetime import datetime, timezone, timedelta
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

print(f"✓ Libraries ready (pandas {pd.__version__}, sklearn, plotly, networkx)")


✓ Libraries ready (pandas 2.3.3, sklearn, plotly, networkx)


## 2. Data ingest

Loads Reddit `.json` and `.jsonl` files from `DATA_DIR`, normalizes posts and comments into a single DataFrame, and **separates content from metadata**. This is the key structural change from earlier runs: downstream NLP only sees `text_content`, never subreddit names, author handles, or URLs.


### 2.1 Low-level loaders and field extractors

Carried forward from EPH (which already handles the three JSON shapes we see in practice — bare list, wrapper dict, JSON Lines — and distinguishes posts from comments by schema).


In [3]:
def load_data_file(filepath):
    """Load .json or .jsonl; tolerates list, {data: [...]}, and line-delimited."""
    with open(filepath, "r", encoding="utf-8") as fh:
        content = fh.read().strip()
    if not content:
        return []
    try:
        data = json.loads(content)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            for key in ("data", "posts", "comments", "items", "results"):
                if key in data and isinstance(data[key], list):
                    return data[key]
            return [data]
    except json.JSONDecodeError:
        pass
    # Fallback: JSON Lines
    return [json.loads(ln) for ln in content.splitlines() if ln.strip()
            and _safe_json_line(ln)]

def _safe_json_line(ln):
    try:
        json.loads(ln); return True
    except Exception:
        return False

def _to_dt(ts):
    if ts is None: return None
    try:
        if isinstance(ts, (int, float)):
            return datetime.fromtimestamp(ts, tz=timezone.utc)
        return datetime.fromisoformat(str(ts).replace("Z", "+00:00"))
    except Exception:
        return None

def _guess_subreddit(fp):
    name = Path(fp).stem.lower()
    for suf in ("_posts", "_comments", "_post", "_comment", "_submissions"):
        name = name.replace(suf, "")
    return name[2:] if name.startswith("r_") else name

def _normalize(rec, fp):
    """Return dict with CONTENT fields and METADATA fields separated."""
    is_comment = "link_id" in rec and "title" not in rec
    title = rec.get("title") or ""
    body  = rec.get("selftext") or rec.get("body") or ""
    if body in ("[removed]", "[deleted]"):
        body = ""
    return {
        # ── content (goes into NLP) ────────────────────────────
        "title":        title,
        "body":         body,
        "text_content": f"{title} {body}".strip(),
        # ── metadata (kept separate from NLP) ──────────────────
        "type":         "comment" if is_comment else "post",
        "post_id":      rec.get("id") or (rec.get("link_id", "").lstrip("t3_")),
        "author":       rec.get("author", "[unknown]"),
        "subreddit":    rec.get("subreddit") or _guess_subreddit(fp),
        "permalink":    "https://reddit.com" + (rec.get("permalink") or ""),
        "timestamp":    _to_dt(rec.get("created_utc") or rec.get("created")),
        "upvote":       rec.get("ups", rec.get("score", 0)),
        "downvote":     rec.get("downs", 0),
        "source_file":  Path(fp).name,
    }

print("✓ Loader helpers defined")


✓ Loader helpers defined


### 2.2 Load all files into a single DataFrame

Progress prints per-file so you can spot failures. Empty text rows are dropped. The resulting `df_raw` has one row per post/comment, content and metadata in distinct columns.


In [4]:
files = sorted(list(DATA_DIR.glob("*.json")) + list(DATA_DIR.glob("*.jsonl")))
print(f"Found {len(files)} files in {DATA_DIR}")

rows = []
for fp in files:
    try:
        recs = load_data_file(fp)
        for r in recs:
            if isinstance(r, dict):
                rows.append(_normalize(r, fp))
        print(f"  ✓ {fp.name}: {len(recs):,} records")
    except Exception as e:
        print(f"  ✗ {fp.name}: {type(e).__name__}: {e}")

df_raw = pd.DataFrame(rows)
df_raw = df_raw[df_raw["text_content"].str.len() > 0].reset_index(drop=True)
print(f"\n✓ df_raw: {len(df_raw):,} rows × {df_raw.shape[1]} cols")
print(f"  Content columns:  ['title', 'body', 'text_content']")
print(f"  Metadata columns: {[c for c in df_raw.columns if c not in ('title','body','text_content')]}")
df_raw.head(2)


Found 34 files in C:\Users\ecm3479\OneDrive - The University of Texas at Austin\Documents\Media Scrubbing Worksheets\JSON
  ✓ r_appalachia_comments.json: 197,366 records
  ✓ r_appalachia_posts.json: 7,098 records
  ✓ r_asheville_comments.json: 827,514 records
  ✓ r_asheville_posts.json: 35,894 records
  ✗ r_austin_comments.json: MemoryError: 
  ✓ r_austin_posts.json: 78,591 records
  ✓ r_climate_comments.json: 217,341 records
  ✓ r_climate_posts.json: 24,697 records
  ✓ r_climatechange_comments.json: 234,520 records
  ✓ r_climatechange_posts.json: 10,958 records
  ✓ r_climateskeptics_comments.json: 53,404 records
  ✓ r_climateskeptics_posts.json: 4,437 records
  ✓ r_collapse_comments.json: 530,179 records
  ✓ r_collapse_posts.json: 14,577 records
  ✓ r_conspiracy_comments.json: 151,254 records
  ✓ r_conspiracy_posts.json: 123,325 records
  ✓ r_environment_comments.json: 99,556 records
  ✓ r_environment_posts.json: 14,772 records
  ✓ r_HurricaneHelene_comments.json: 9,530 records
  ✓ r_

,title,body,text_content,type,post_id,author,subreddit,permalink,timestamp,upvote,downvote,source_file
0,,"Exactly! Well, as the map seems to show a west...","Exactly! Well, as the map seems to show a west...",comment,lot6fqs,thehorselesscowboy,Appalachia,https://reddit.com/r/Appalachia/comments/1fotm...,2024-09-25 05:01:25+00:00,4,0,r_appalachia_comments.json
1,,Please don’t try and drive through rushing water,Please don’t try and drive through rushing water,comment,lotfjwm,Summoorevincent,Appalachia,https://reddit.com/r/Appalachia/comments/1foo1...,2024-09-25 06:31:46+00:00,2,0,r_appalachia_comments.json


## 3. Boolean corpus partitioning

This is the EPH logic preserved verbatim in structure, but applied in vectorized form. Each document ends up tagged with every boolean search it matches, so downstream sections can slice the corpus by `Flood_GovResp`, `LAfires_Misinformation`, etc.


### 3.1 Keyword groups and boolean searches

Edit these in place to change the analysis. `_GROUP_REGISTRY` is the building block; `BOOLEAN_SEARCHES` composes groups with AND/OR logic. The list of `requirements` is AND-joined, and a nested list is an OR-joined alternative.


In [7]:
_GROUP_REGISTRY = {
    "NC":      ["NC", "north carolina","western carolina", "WNC", "western appalachia", "appalachia", "buncombe co", "BC", "catawba co", 
                "rutherford co", "haywood co", "asheville", "Montreat", "Chimney Rock", "cherokee", "Sep 26", "september 26", "9/26", "9-26"],
    
    "TX":      ["TX", "Texas", "CTX", "central texas", "hill country", "kerr co", "kendall co", "travis co", "kerrville", "comfort", "hunt", "ingram", 
               "center point", "san saba", "menard", "mason", "bumble bee hills", "july 4", "7/4", "7-4"],
    
    "Flood":   ["flood", "flash flood", "inundation","wall of water", "tropical storm","inland tsunami", "flooding", "floodplain", "hurricane", "guadalupe", "Cypress creek", 
                "north fork", "south fork", "edmunson creek", "flash flood alley","Swannanoa", "French Broad", "catawba", "winkler creek", "helene"],
    
    "Radar":  ["Rainmaker", "severe rainfall", "severe thunderstorm", "weather radar system", "weather station", "weather forecast", "category 4", 
               "category four", "100-year", "500-year", "alert", "warning", "watch", "notification", "message", "WARN-CTX", "notification", "IPAWS"],
    
    "Hoax":   ["conspiracy", "misinformation", "disinformation","fake", "false", "inaccurate", "wrong", "lying", "rumor", "lie", "misleading", 
               "propaganda", "rhetoric", "mainstream", "anti-american","geo-engineered weapons", "weather weapon", "cloud seeding", 
               "weather manipulation", "weather machine","climate hoax", "control the weather", "AI-generated", "AI slop", "scam", "cherokee indian", "cherokee tribe",
               "migrants", "God-like", "mother nature", "God's plan", "nothing they could have done", 
               "help delayed", "those little girls are dead", "weird cult", "food confiscation", "never happened before"],
    
    "Infra":  ["I-40", "I-26", "I-39", "impassable", "power outage","lost power", "no power","lack of cell service", "no internet connection", 
               "no signal", "no cell service", "no water", "burst pipe", "boil water advisory","wastewater overflow", "SSO", "sewer overflow", 
               "sewer runoff", "bridge", "low-water crossing"],
    
    "Natsys": ["landslide", "erosion", "debris", "cypress tree", "damming", "stacking up", "water contamination", "water quality", 
               "sediment buildup", "vegetation"],
    
    "Health": ["PTSD", "chronic stress","heart attack", "suicide", "mold", "mildew", "disease", "illness", "infection", "asthma", "premature death", "anxiety", "depression", "hospital",
               "healthcare desert", "IV shortage", "displacement"],
    
    "Response": ["evacuation", "evacuate", "cutting off access", "trapped", "emergency response", "first responder", "search and rescue", 
               "missing person", "home cleanup", "municipal repair","no helicopters", "no rescue", "firefighter", "fire department",
               "game warden", "volunteer", "state of emergency", "disaster declaration"],
    #advice on how to separate agencies, government titles, and specific people...it is separating the boolean groups as well but I want to make
    #sure to include references to one's title separate from name, but also with name, but having "kelly" or "trump" in there will sckew results too
    "Agency": ["FEMA", "NOAA", "NWS", "National Weather Service", "TDEM", "TPWD", "TWDB", "TxDOT", "NCEM", "DEQ", "NCORR",
               "DPS", "UGRA", "GBRA", "FDNY", "DOGE"],
    
    "Gov":    ["president", "former president","administration", "lieut gov", "lieutenant governor", "governor", "rep", 
               "representative", "senator", "chairman", "mayor", "director", "judge", "liberal",
               "leftist", "extremist", "conservative", "republican", "democrat", "moderate", "maga"],
    
    "Govppl": ["joe biden","donald trump", "biden", "trump", "Chuck Edwards", "Edwards", "Mark Robinson", "robinson", "josh stein", 
               "roy cooper", "Gov Cooper", " Gov Abbott", "Greg Abbott", "Abbott","ray howard", "stolarczyk", "joe herring", "dub thomas", 
               "william thomas", "larry leitha", "tom moser","rob kelly", "kelly", "william rector", "king", "donna campbell", "campbell", 
               "flores", "wes virdell", "virdell", "eastland"],
    
    "GovAid":  ["disaster relief","blockade", "relief aid", "federal aid", "SBA loan","funding", "apply", "application", 
                "TSA Program","money", "flood insurance", "LOMR", "LOMA", "national cuts", "budget cuts", "waste"],

    "Media":  [ "media", "online", "podcast", "X", "twitter", "retweet", "X Community", "chatroom", "insta", "instagram", "instagram group", 
                "instagram post", "subreddit post", "reddit thread", "subreddit thread", "truth social","YouTube", "fb", "facebook group", "facebook post", "meta",
                "alex jones", "joe rogan", "charlie kirk", "shellenberger", "kirk", "rogan", "influencer", "super spreader", "clout chaser"],
    
    "News":   ["fox", "fox news", "new york times", "NYT", "CNN", "abc", "nbc", "npr", "cbs", "breitbart", "national review", 
               "washington times", "turning point", "prageru", "news"],
    
    "Tourist": ["la junta", "Mo Ranch", "camp mystic", "Heart O' the Hills", "mystic cabin", "senior hill", "bubble inn", "eastland", 
                "Mystic Cypress Lake", "Heaven's 27", "mystic lawsuit", "tourist", "outsider", "RV parks", "Blue Oak RV", "casa bonita"],
}
#These are all separating the results too much, I would like it condensed by topic & event but with how I have the terms setup, im not sure what to do
BOOLEAN_SEARCHES = [
    #General Themes for Science Backbone network
    {"name": "GWTX",   "event": "TX_Flood",     "requirements": ["TX","Flood", "Radar",["Gov", "Govppl", "Agency"]]},
    {"name": "GWNC",   "event": "NC_Helene",   "requirements": ["NC","Flood", "Radar",["Gov", "Govppl", "Agency"]]},
    {"name": "MISTX",   "event": "TX_Flood",     "requirements": ["TX", ["Flood", "Radar"],"Hoax", ["Media", "News", "Gov", "Govppl", "Agency"]]},
    {"name": "MISNC",   "event": "NC_Helene",     "requirements": ["NC", ["Flood", "Radar"],"Hoax", ["Media", "News", "Gov", "Govppl", "Agency"]]},
    {"name": "FFDTX",   "event": "TX_Flood",   "requirements": ["TX", "Flood", ["Infra", "Natsys"]]},
    {"name": "FDNC",   "event": "NC_Helene",   "requirements": ["NC", "Flood", ["Infra", "Natsys"]]},
    {"name": "GRTX",           "event": "TX_Flood",     "requirements": ["TX","Flood", ["Infra", "Natsys", "Response"], ["Gov", "Agency", "Govppl"]]},
    {"name": "GRNC",           "event": "NC_Helene",     "requirements": ["NC","Flood", ["Infra", "Natsys", "Response"], ["Gov", "Agency", "Govppl"]]},
    {"name": "TourTX",           "event": "TX_Flood",     "requirements": ["TX","Flood", "Tourist"]},
    {"name": "TourNC",           "event": "NC_Helene",     "requirements": ["NC","Flood", "Tourist"]},
    {"name": "EPHTX",   "event": "TX_Flood",   "requirements": ["TX", ["Flood", "Radar"], "Health"]},
    {"name": "EPHNC",   "event": "NC_Helene",   "requirements": ["NC", ["Flood", "Radar"], "Health"]},
    #More specific searches for Jaccard and event-specific backbone network
    #Tropical Storm Helene
    {"name": "GOWNC",   "event": "NC_Helene",   "requirements": ["NC","Flood", "Radar", "Gov", "Govppl"]},
    {"name": "AWNC",   "event": "NC_Helene",   "requirements": ["NC","Flood", "Radar","Agency"]},
    {"name": "GovMisNC",   "event": "NC_Helene",     "requirements": ["NC", "Flood", "Radar", "Hoax", ["Gov", "Govppl", "Agency"]]},
    {"name": "MediaMisNC",   "event": "NC_Helene",     "requirements": ["NC", "Media", "Hoax", ["Flood", "Radar"]]},
    {"name": "NewsMisNC",   "event": "NC_Helene",     "requirements": ["NC", "News", "Hoax", ["Flood", "Radar"]]},
    {"name": "GovAidNC",   "event": "NC_Helene",   "requirements": ["NC","Flood", ["Infra", "Natsys"], ["GovAid", "Response"]]},
    {"name": "RespMediaNC",   "event": "NC_Helene",   "requirements": ["NC","Flood", ["GovAid", "Response"], ["Gov", "Govppl", "Agency"], ["News", "Media"]]},
    {"name": "RespMisNC",  "event": "NC_Helene",  "requirements": ["NC","Flood", ["Response", "GovAid"], "Hoax"]},
    {"name": "DisMisNC",  "event": "NC_Helene",  "requirements": ["NC","Flood", ["Infra", "Natsys", "Tourist"], "Hoax"]},
    {"name": "EPHImpactsNC",   "event": "NC_Helene",   "requirements": ["NC", "Flood", "Health", ["Infra", "Natsys", "Response", "GovAid"]]},
    {"name": "EPHMisNC",   "event": "NC_Helene",   "requirements": ["NC", "Flood", "Health", "Hoax"]},
    #Texas Floods CTX Floods
    {"name": "GOWTX",   "event": "TX_Flood",     "requirements": ["TX","Flood", "Radar","Gov", "Govppl"]},
    {"name": "AWTX",   "event": "TX_Flood",   "requirements": ["TX","Flood", "Radar","Agency"]},
    {"name": "GovMisTX",   "event": "TX_Flood",     "requirements": ["TX", "Flood", "Radar", "Hoax", ["Gov","Govppl", "Agency"]]},
    {"name": "MediaMisTX",   "event": "TX_Flood",     "requirements": ["TX", "Media", "Hoax", ["Flood", "Radar"]]},
    {"name": "NewsMisTX",   "event": "TX_Flood",     "requirements": ["TX", "News", "Hoax", ["Flood", "Radar"]]},
    {"name": "GovAidTX",   "event": "TX_Flood",   "requirements": ["TX", "Flood", ["Infra", "Natsys"], ["GovAid", "Response"]]},
    {"name": "RespMediaTX",   "event": "TX_Flood",   "requirements": ["TX", "Flood", ["GovAid", "Response"], ["Gov", "Govppl", "Agency"], ["News", "Media"]]},
    {"name": "RespMisTX",  "event": "TX_Flood",  "requirements": ["TX","Flood", ["Response", "GovAid"], "Hoax"]},
    {"name": "DisMisTX",  "event": "TX_Flood",  "requirements": ["TX","Flood", ["Infra", "Natsys", "Tourist"], "Hoax"]},
    {"name": "EPHImpactsTX",   "event": "TX_Flood",   "requirements": ["TX", "Flood", "Health", ["Infra", "Natsys", "Response", "GovAid"]]},
    {"name": "EPHMisTX",   "event": "TX_Flood",   "requirements": ["TX", "Flood", "Health", "Hoax"]},


]

print(f"✓ {len(_GROUP_REGISTRY)} keyword groups, {len(BOOLEAN_SEARCHES)} boolean searches")


✓ 16 keyword groups, 34 boolean searches


### 3.2 Vectorized boolean matcher

Each row of `df_raw` gets a tag column per boolean search (True/False). A document can match multiple searches (e.g., a post about FEMA response in NC hits both `Flood_GovResp` and `HELENE_Misinformation` if it also mentions conspiracy terms). This multi-tag approach is what enables the cross-filter comparisons in Section 7.


In [8]:
def _any_term_hit(text_lower, group_or_term):
    """Resolve a requirement token to a flat term list and test ANY-match."""
    if isinstance(group_or_term, list):
        terms = []
        for x in group_or_term:
            terms.extend(_GROUP_REGISTRY.get(x, [x]) if isinstance(x, str) else [x])
    elif group_or_term in _GROUP_REGISTRY:
        terms = _GROUP_REGISTRY[group_or_term]
    else:
        terms = [group_or_term]
    return any(t.lower() in text_lower for t in terms)

def _matches_boolean(text, bs):
    tl = text.lower()
    return all(_any_term_hit(tl, req) for req in bs["requirements"])

# Tag each document with every boolean search it matches
for bs in BOOLEAN_SEARCHES:
    df_raw[f"bool_{bs['name']}"] = df_raw["text_content"].map(lambda t, _bs=bs: _matches_boolean(t, _bs))

# Compact summary columns: which searches match, which event(s)
bool_cols = [f"bool_{bs['name']}" for bs in BOOLEAN_SEARCHES]
df_raw["boolean_matches"] = df_raw[bool_cols].apply(
    lambda r: [bs["name"] for bs, hit in zip(BOOLEAN_SEARCHES, r) if hit], axis=1)
df_raw["events"] = df_raw["boolean_matches"].apply(
    lambda names: sorted({bs["event"] for bs in BOOLEAN_SEARCHES if bs["name"] in names and bs["event"] != "Multi"}))
df_raw["any_match"] = df_raw["boolean_matches"].str.len() > 0

# Filter to matched subset for downstream analysis
df = df_raw[df_raw["any_match"]].copy().reset_index(drop=True)
match_summary = pd.DataFrame({
    "boolean_search": [bs["name"] for bs in BOOLEAN_SEARCHES],
    "event":          [bs["event"] for bs in BOOLEAN_SEARCHES],
    "matches":        [int(df_raw[f"bool_{bs['name']}"].sum()) for bs in BOOLEAN_SEARCHES],
})
print(f"✓ {len(df):,} of {len(df_raw):,} docs matched at least one boolean search")
match_summary


✓ 40,037 of 3,419,870 docs matched at least one boolean search


,boolean_search,event,matches
0,GWTX,TX_Flood,1067
1,GWNC,NC_Helene,6902
2,MISTX,TX_Flood,3370
3,MISNC,NC_Helene,27548
4,FFDTX,TX_Flood,522
5,FDNC,NC_Helene,3912
6,GRTX,TX_Flood,861
7,GRNC,NC_Helene,5445
8,TourTX,TX_Flood,938
9,TourNC,NC_Helene,2233


## 4. Topic analysis seeded by the boolean searches

**Key conceptual shift**: in the old Load & Bridge flow, LDA ran over the entire corpus and its topics frequently reflected metadata artifacts (subreddit names, author handles, formatting boilerplate). Here the boolean searches are the *primary* topics — they are researcher-defined, grounded in decision theory, and named. LDA is used only *within each boolean sub-corpus* to find sub-themes. Results stay focused on the boolean partitions by construction.


### 4.1 Content preprocessing with metadata-aware stopwords

We strip Reddit and platform boilerplate (`http`, `www`, `edit`, `deleted`, etc.) plus the standard English stopword list plus every subreddit and author name in the corpus (those are metadata leaking into content). We also drop the boolean keyword terms themselves from the stopword list — you *want* those to remain prominent because they define the topic.


In [9]:
STOPWORDS_BASE = {
    # Standard English (compact subset)
    "a","about","above","after","again","all","am","an","and","any","are","as","at",
    "be","because","been","before","being","below","between","both","but","by","can",
    "did","do","does","doing","don","down","during","each","few","for","from","further",
    "had","has","have","having","he","her","here","hers","herself","him","himself","his",
    "how","i","if","in","into","is","it","its","itself","just","let","me","more","most",
    "my","myself","no","nor","not","now","of","off","on","once","only","or","other","our",
    "ours","ourselves","out","over","own","s","same","she","should","so","some","such","t",
    "than","that","the","their","theirs","them","themselves","then","there","these","they",
    "this","those","through","to","too","under","until","up","very","was","we","were",
    "what","when","where","which","while","who","whom","why","will","with","you","your",
    "yours","yourself","yourselves",
    # Reddit / platform artifacts
    "http","https","www","com","reddit","imgur","edit","deleted","removed","nbsp","amp",
    "like","know","think","going","really","would","could","got","get","just",
    "yeah","okay","um","uh","oh","actually","basically","literally", "lol",
}

def build_stopwords(df):
    sw = set(STOPWORDS_BASE)
    # Subreddit and author names are metadata leakage — drop them from content vocab
    sw.update(str(s).lower() for s in df["subreddit"].dropna().unique())
    sw.update(str(a).lower().lstrip("u_") for a in df["author"].dropna().unique())
    sw.discard("")
    return sw

CONTENT_STOPWORDS = build_stopwords(df)
print(f"✓ {len(CONTENT_STOPWORDS):,} stopwords (incl. {df['subreddit'].nunique()} subreddits, {df['author'].nunique()} authors)")


✓ 15,921 stopwords (incl. 17 subreddits, 15748 authors)


### 4.2 LDA sub-topics within each boolean sub-corpus

For every boolean search with a reasonable sample (≥25 documents), we run a small LDA with 3 sub-topics. Sub-corpora smaller than that get a single TF-IDF "top-terms" summary instead — LDA is unreliable at small N. Results land in a tidy DataFrame `subtopics_df` with columns: `boolean_search`, `subtopic_id`, `top_terms`, `n_docs`.


In [10]:
SUBTOPICS_PER_CORPUS = 3
MIN_DOCS_FOR_LDA = 25

def top_terms_tfidf(texts, stopwords, n=10):
    vec = TfidfVectorizer(max_features=2000, stop_words=list(stopwords),
                          ngram_range=(1,2), min_df=2)
    try:
        X = vec.fit_transform(texts)
    except ValueError:
        return []
    means = np.asarray(X.mean(axis=0)).ravel()
    vocab = np.array(vec.get_feature_names_out())
    top_idx = means.argsort()[::-1][:n]
    return list(vocab[top_idx])

def lda_subtopics(texts, stopwords, k, n_terms=10):
    vec = CountVectorizer(max_features=2000, stop_words=list(stopwords),
                          ngram_range=(1,2), min_df=2)
    try:
        X = vec.fit_transform(texts)
    except ValueError:
        return []
    lda = LatentDirichletAllocation(n_components=k, random_state=42,
                                    learning_method="batch", max_iter=20)
    lda.fit(X)
    vocab = np.array(vec.get_feature_names_out())
    return [list(vocab[comp.argsort()[::-1][:n_terms]]) for comp in lda.components_]

subtopic_rows = []
for bs in BOOLEAN_SEARCHES:
    sub = df[df[f"bool_{bs['name']}"]]
    if len(sub) == 0:
        continue
    if len(sub) >= MIN_DOCS_FOR_LDA:
        topic_terms_list = lda_subtopics(sub["text_content"].tolist(),
                                         CONTENT_STOPWORDS, SUBTOPICS_PER_CORPUS)
        for i, terms in enumerate(topic_terms_list):
            subtopic_rows.append({"boolean_search": bs["name"], "event": bs["event"],
                                  "subtopic_id": i, "n_docs": len(sub),
                                  "method": "LDA", "top_terms": terms})
    else:
        subtopic_rows.append({"boolean_search": bs["name"], "event": bs["event"],
                              "subtopic_id": 0, "n_docs": len(sub),
                              "method": "TF-IDF", "top_terms":
                                  top_terms_tfidf(sub["text_content"].tolist(), CONTENT_STOPWORDS)})

subtopics_df = pd.DataFrame(subtopic_rows)
subtopics_df["top_terms_str"] = subtopics_df["top_terms"].str.join(", ")
print(f"✓ {len(subtopics_df)} sub-topics across {subtopics_df['boolean_search'].nunique()} boolean searches")
subtopics_df[["boolean_search", "event", "subtopic_id", "n_docs", "method", "top_terms_str"]]


c:\Users\ecm3479\AppData\Local\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['001', '0121', '02', '0232', '0_tim', '10', '1022', '1059', '1075', '10778', '1078', '108', '1082', '1094', '11', '110', '1105', '1111', '1124', '1127', '1143', '117', '1172', '1181', '1188', '119', '120', '1215', '123', '1234', '1250', '1269', '1270', '13', '131', '1311', '1312', '1313', '1330', '1332', '134', '1349', '1354', '136', '1367', '137', '1385', '1391', '13ender', '1432', '14331', '1466', '1470', '1479', '148', '1485', '1498', '1502', '1509', '151', '1519', '1526', '1540', '155', '156', '1564', '1568', '1578', '1592', '1594', '1597', '1615', '1621', '1622', '1624', '1642', '1653', '1670', '1678', '168', '17', '171', '1726', '1728', '1758', '1799', '1821', '1859', '1869', '187', '1875', '1899', '19', '1908', '191', '1912', '1933', '1934', '1945', '1980', '19890',

✓ 102 sub-topics across 34 boolean searches


,boolean_search,event,subtopic_id,n_docs,method,top_terms_str
0,GWTX,TX_Flood,0,1067,LDA,"2025, org, archive, ph, archive ph, world, las..."
1,GWTX,TX_Flood,1,1067,LDA,"flood, camp, river, county, mystic, flooding, ..."
2,GWTX,TX_Flood,2,1067,LDA,"people, one, re, time, us, even, said, trump, ..."
3,GWNC,NC_Helene,0,6902,LDA,"people, flood, one, water, time, 2025, also, u..."
4,GWNC,NC_Helene,1,6902,LDA,"megathread, please, information, questions, ge..."
...,...,...,...,...,...,...
97,EPHImpactsTX,TX_Flood,1,154,LDA,"people, one, time, even, world, years, many, u..."
98,EPHImpactsTX,TX_Flood,2,154,LDA,"2025, archive, ph, archive ph, org, world, las..."
99,EPHMisTX,TX_Flood,0,176,LDA,"one, people, god, world, flood, even, time, wa..."
100,EPHMisTX,TX_Flood,1,176,LDA,"2025, archive, ph, archive ph, org, world, las..."


### 4.3 Optional: LLM topic labeling (Claude) with expert-triage flag

This cell is an **optional enhancement** that asks Claude to turn each sub-topic's top terms plus three exemplar snippets into a short human-readable label (e.g., `"Palisades evacuation logistics"` instead of `"palisades|evacuation|fire|mandatory|order|order issued"`). Useful for publication tables and stakeholder-facing reports.

**Architecture note — verifiability-first.** Labels generated here are flagged `provisional` and never overwrite anything. They live alongside the raw `top_terms` column, which remains the ground-truth anchor. A human reviewer sets `expert_triaged = True` after validation. This is deliberate: the LLM is a labor-saving helper, not the epistemic source.

#### How to enable on TACC

1. Set the environment variable `ANTHROPIC_API_KEY` in your TACC shell before launching Jupyter. In a login or compute node:
   ```bash
   export ANTHROPIC_API_KEY="sk-ant-..."
   ```
   Put this in your `~/.bashrc` or a `.env` file the notebook loads, not directly in the notebook (you don't want to commit keys). On TACC specifically, storing it in `$WORK/.env` and reading with `python-dotenv` is a common pattern.
2. Flip `ENABLE_LLM_LABELS = True` in the cell below.
3. Confirm the model name in `LLM_MODEL` matches what's available. Current default is `claude-opus-4-7`; check `docs.claude.com` for the current model roster if you want to pin to a specific Sonnet or Haiku version for cost or speed reasons.
4. Re-run the cell. Each sub-topic costs roughly one cheap LLM call (~200 input tokens + ~20 output tokens). For the typical 15–20 sub-topics in this notebook, the full labeling pass finishes in well under a minute.

#### What happens when the LLM is NOT available (no key, network blocked, quota exhausted, or you just don't want to use it)

Leave `ENABLE_LLM_LABELS = False`. The cell runs harmlessly as a no-op and the pipeline continues. Downstream sections (5, 6, 7, 8) do **not** depend on LLM labels — every figure and export uses `top_terms` and `boolean_search` as the identifiers, which are always populated. You lose only the readable labels in the final report; sub-topics are still named by their boolean parent (e.g., `LAfires_GovResp_subtopic_0`) and their top terms.

If you want readable labels without the LLM, three easy manual alternatives:

- **Just read the top_terms.** For a 15-row `subtopics_df` this is a 5-minute visual scan and produces labels the expert actually trusts.
- **Spreadsheet triage.** Open `subtopics_{timestamp}.csv` (written in Section 8) in Excel, add a column named `manual_label`, fill it in, and re-import. This scales well if you're doing this with a research assistant.
- **Rule-based labeling.** Pick the most specific 2–3 top_terms per sub-topic and concatenate them. Crude but reproducible.


In [11]:
ENABLE_LLM_LABELS = False          # ← flip to True after setting ANTHROPIC_API_KEY
LLM_MODEL         = "claude-opus-4-7"

def llm_label_topic(top_terms, examples):
    import os
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("ANTHROPIC_API_KEY is not set in the environment.")
    try:
        from anthropic import Anthropic
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anthropic"])
        from anthropic import Anthropic
    client = Anthropic()   # reads ANTHROPIC_API_KEY from env
    prompt = (
        "You are labeling a sub-topic extracted from a Reddit corpus about natural disaster events.\n"
        f"Top terms: {', '.join(top_terms)}\n\n"
        "Three example snippets:\n" + "\n---\n".join(examples[:3]) +
        "\n\nReturn ONLY a 3–7 word topic label. No preamble, no quotation marks."
    )
    resp = client.messages.create(model=LLM_MODEL, max_tokens=40,
                                  messages=[{"role": "user", "content": prompt}])
    return resp.content[0].text.strip()

subtopics_df["llm_label_provisional"] = ""
subtopics_df["expert_triaged"] = False

if ENABLE_LLM_LABELS:
    labels = []
    for _, row in subtopics_df.iterrows():
        sub = df[df[f"bool_{row['boolean_search']}"]]
        examples = sub["text_content"].head(3).tolist()
        try:
            labels.append(llm_label_topic(row["top_terms"], examples))
        except Exception as e:
            labels.append(f"[label-failed: {type(e).__name__}]")
    subtopics_df["llm_label_provisional"] = labels
    print(f"✓ Generated {sum(1 for l in labels if not l.startswith('['))} LLM labels "
          f"(flagged 'provisional' pending expert triage)")
else:
    print("  LLM labeling disabled — top_terms will serve as sub-topic identifiers downstream.")
    print("  To enable: set ANTHROPIC_API_KEY and flip ENABLE_LLM_LABELS = True (see markdown above).")

subtopics_df[["boolean_search", "subtopic_id", "top_terms_str", "llm_label_provisional"]].head(10)


  LLM labeling disabled — top_terms will serve as sub-topic identifiers downstream.
  To enable: set ANTHROPIC_API_KEY and flip ENABLE_LLM_LABELS = True (see markdown above).


,boolean_search,subtopic_id,top_terms_str,llm_label_provisional
0,GWTX,0,"2025, org, archive, ph, archive ph, world, las...",
1,GWTX,1,"flood, camp, river, county, mystic, flooding, ...",
2,GWTX,2,"people, one, re, time, us, even, said, trump, ...",
3,GWNC,0,"people, flood, one, water, time, 2025, also, u...",
4,GWNC,1,"megathread, please, information, questions, ge...",
5,GWNC,2,"helene, hurricane helene, please, hurricane, r...",
6,MISTX,0,"trump, events, government, public, state, soci...",
7,MISTX,1,"flood, camp, 2025, water, org, year, mystic, r...",
8,MISTX,2,"people, one, time, even, re, us, also, ve, way...",
9,MISNC,0,"people, one, us, time, even, re, also, world, ...",


## 5. Science backbone mapping and network visualization

Preserves the Load & Bridge logic of mapping topics to an ETO Map-of-Science-style backbone of domains and subdisciplines, but the network is now overlaid **per boolean search** rather than over a single undifferentiated corpus. Each boolean search gets its own colored trace in the network, so you can see at a glance that (for example) LA Fires light up Atmospheric Science + Public Health while Helene lights up Hydrology + Infrastructure Engineering.


### 5.1 Science backbone — three-tier resolution

The backbone is the domain/subdiscipline structure that discovered sub-topics get mapped onto. The notebook tries three sources, in order of preference, and uses the first one that works:

| Tier | Source | When it's used |
|---|---|---|
| 1 | ETO Map-of-Science CSV export | You've set `ETO_EXPORT_PATH` to a real CSV file |
| 2 | `semantic_bridge_pipeline.default_science_backbone()` | The `sbp` package is installed in the environment (normal TACC case) |
| 3 | Inlined stand-in (defined in this cell) | Neither of the above — typically a laptop run or a fresh environment without the TACC tutorial package |

**About the ETO export (Tier 1).** The [Emerging Technology Observatory Map of Science](https://eto.tech/tool/map-of-science/) is a large clustered network of research disciplines maintained by Georgetown CSET. Exports aren't auto-downloaded; you generate them by running queries in the ETO web tool and clicking "Export CSV." Section 5 of the prior Load & Bridge notebook includes a cell that produces suggested query URLs for your discovered topics — that's still the right workflow if you want the richest backbone. Expected CSV columns: `domain`, `subdiscipline`, `keywords`.

**About the sbp default (Tier 2).** The `semantic_bridge_pipeline` package ships a curated default backbone tuned for the TACC tutorial datasets. This is what the prior Load & Bridge notebook used when an ETO export wasn't present. If you're running on TACC next to the tutorial materials, this tier will fire automatically.

**About the inlined stand-in (Tier 3).** If neither `ETO_EXPORT_PATH` nor the `sbp` package is available, the notebook falls back to a compact 8-domain backbone defined directly in the cell below. It's intentionally small and domain-tuned for the three disaster events — not a replacement for the full ETO map, but sufficient for self-contained development runs and for getting the pipeline going before you've set up ETO or `sbp`. The cell prints exactly which tier it resolved to so you know at a glance.


In [12]:
# Set this to a real ETO CSV path to force Tier 1. Leave as None to skip Tier 1.
ETO_EXPORT_PATH = None

SCIENCE_BACKBONE = None
BACKBONE_SOURCE  = None

# ── Tier 1: ETO Map-of-Science CSV export ─────────────────────────────
if ETO_EXPORT_PATH and Path(ETO_EXPORT_PATH).exists():
    try:
        import semantic_bridge_pipeline as _sbp
        _eto_df = _sbp.load_eto_cluster_export(ETO_EXPORT_PATH)
        _candidate = _sbp.build_science_backbone_from_eto_export(_eto_df)
        if _candidate:
            SCIENCE_BACKBONE = _candidate
            BACKBONE_SOURCE  = f"Tier 1: ETO export ({ETO_EXPORT_PATH})"
    except Exception as e:
        print(f"  ⚠ ETO export present but failed to load: {type(e).__name__}: {e}")

# ── Tier 2: semantic_bridge_pipeline default ──────────────────────────
if SCIENCE_BACKBONE is None:
    try:
        import semantic_bridge_pipeline as _sbp
        _candidate = _sbp.default_science_backbone()
        if _candidate:
            SCIENCE_BACKBONE = _candidate
            BACKBONE_SOURCE  = "Tier 2: semantic_bridge_pipeline.default_science_backbone()"
    except ImportError:
        pass

# ── Tier 3: inlined stand-in (self-contained fallback) ────────────────
if SCIENCE_BACKBONE is None:
    SCIENCE_BACKBONE = {
        "Atmospheric Science":      {"subdisciplines": ["Meteorology", "Climate Science", "Severe Weather"],
                                      "terms": ["wind","storm","precipitation","forecast","climate","atmosphere","warming", "NOAA"]},
        "Riverine Flash Flooding":  {"subdisciplines": ["Flood Hydrology", "Water Resources", "Debris Damage"],
                                      "terms": ["flood","river","water","rainfall","runoff","basin","watershed","dam","levee", "landslide", 
                                                "mudslide", "drought", "erosion", "debris", "guadalupe", "fork", "french broad", "swannanoa"]},
        "Emergency Management":     {"subdisciplines": ["Response", "Evacuation", "Recovery"],
                                      "terms": ["evacuation","rescue","relief","emergency","response","shelter","FEMA","aid"]},
        "Public Health":            {"subdisciplines": ["Environmental Health", "Disaster Mental Health"],
                                      "terms": ["health","injury","mold","water quality","mental","trauma","grief", "anxiety", "contamination"]},
        "Infrastructure Engineering":{"subdisciplines": ["Power", "Transportation", "Communications"],
                                      "terms": ["power","bridge","road","low-water crossing", "network","outage","cell",
                                                "internet", "infrastructure","repair", "access"]},
        "Political Communication":  {"subdisciplines": ["Government Messaging", "Social Media Discourse"],
                                      "terms": ["president","governor","administration","biden","trump","FEMA","abbott","NWS", "liberal",
                                               "maga", "republican", "UGRA"]},
        "Information Integrity":    {"subdisciplines": ["Misinformation", "Disinformation"],
                                      "terms": ["conspiracy","hoax","fake","misinformation","disinformation","weather weapon","cloud seeding"]},
    }
    BACKBONE_SOURCE = ("Tier 3: inlined stand-in "
                       "(neither ETO_EXPORT_PATH nor semantic_bridge_pipeline available)")

# Defensive mapper — tolerates both the inlined shape {'subdisciplines', 'terms'}
# and variants that sbp / ETO might return (e.g., 'subs' / 'keywords')
def map_terms_to_backbone(terms, backbone):
    terms_l = [t.lower() for t in terms]
    hits = []
    for domain, node in backbone.items():
        if not isinstance(node, dict):
            continue
        subs = node.get("subdisciplines") or node.get("subs") or ["General"]
        node_terms = node.get("terms") or node.get("keywords") or [domain.lower()]
        for sub in subs:
            score = sum(1 for t in terms_l if any(str(kw).lower() in t for kw in node_terms))
            if score > 0:
                hits.append((domain, sub, score))
    if not hits:
        return [("Uncategorized", "General", 0)]
    return sorted(hits, key=lambda x: -x[2])[:3]

subtopics_df["backbone_mapping"] = subtopics_df["top_terms"].apply(
    lambda ts: map_terms_to_backbone(ts, SCIENCE_BACKBONE))
subtopics_df["primary_domain"] = subtopics_df["backbone_mapping"].apply(lambda h: h[0][0])

print(f"✓ {BACKBONE_SOURCE}")
print(f"  {len(SCIENCE_BACKBONE)} domains loaded")
subtopics_df.groupby("primary_domain").size().sort_values(ascending=False)


✓ Tier 3: inlined stand-in (neither ETO_EXPORT_PATH nor semantic_bridge_pipeline available)
  7 domains loaded


primary_domain
Uncategorized              42
Riverine Flash Flooding    37
Political Communication    18
Emergency Management        5
dtype: int64

### 5.2 Network visualization — domains, subdisciplines, and boolean searches

Nodes: science domains (blue), subdisciplines (gray), and boolean searches (colored by event). Edges connect each boolean search to the science backbone nodes its sub-topics map to, with edge width proportional to how many sub-topics land on that node. Hover to see details; the figure renders inline and is also saved to `OUTPUT_DIR` as an HTML file.


In [13]:
def _node_subs(node):
    """Extract subdisciplines list from a backbone node, handling either shape."""
    if not isinstance(node, dict):
        return ["General"]
    return node.get("subdisciplines") or node.get("subs") or ["General"]

def build_network(subtopics_df, backbone):
    G = nx.Graph()
    # Add backbone nodes
    for domain, node in backbone.items():
        G.add_node(domain, kind="domain")
        for sub in _node_subs(node):
            G.add_node(sub, kind="subdiscipline")
            G.add_edge(domain, sub, weight=1)
    # Add boolean-search nodes and connect to subdisciplines via sub-topic mappings
    for bs_name, grp in subtopics_df.groupby("boolean_search"):
        event = grp["event"].iloc[0]
        G.add_node(bs_name, kind="boolean", event=event)
        for _, row in grp.iterrows():
            for domain, sub, score in row["backbone_mapping"]:
                if sub in G:
                    w = G[bs_name].get(sub, {}).get("weight", 0) + score
                    G.add_edge(bs_name, sub, weight=w)
    return G

G = build_network(subtopics_df, SCIENCE_BACKBONE)
pos = nx.spring_layout(G, seed=42, k=0.8, iterations=100)

EVENT_COLORS = {"TX_Flood": "#BF5700", "NC_Helene": "#005f73", "LA_Fires": "#c1121f", "Multi": "#6a4c93"}
KIND_COLORS  = {"domain": "#1f77b4", "subdiscipline": "#8c8c8c"}

edge_x, edge_y = [], []
for u, v in G.edges():
    edge_x += [pos[u][0], pos[v][0], None]
    edge_y += [pos[u][1], pos[v][1], None]

traces = [go.Scatter(x=edge_x, y=edge_y, mode="lines",
                     line=dict(color="#d0d0d0", width=0.8), hoverinfo="none", showlegend=False)]

for kind in ["domain", "subdiscipline"]:
    nodes = [n for n, d in G.nodes(data=True) if d.get("kind") == kind]
    traces.append(go.Scatter(
        x=[pos[n][0] for n in nodes], y=[pos[n][1] for n in nodes], mode="markers+text",
        marker=dict(size=22 if kind=="domain" else 14, color=KIND_COLORS[kind], line=dict(color="white", width=1)),
        text=nodes, textposition="top center", textfont=dict(size=9),
        hoverinfo="text", name=kind.capitalize()))

for event, color in EVENT_COLORS.items():
    nodes = [n for n, d in G.nodes(data=True) if d.get("kind")=="boolean" and d.get("event")==event]
    if not nodes: continue
    traces.append(go.Scatter(
        x=[pos[n][0] for n in nodes], y=[pos[n][1] for n in nodes], mode="markers+text",
        marker=dict(size=18, color=color, symbol="diamond", line=dict(color="white", width=1)),
        text=nodes, textposition="bottom center", textfont=dict(size=9, color=color),
        hoverinfo="text", name=f"Boolean: {event}"))

fig_network = go.Figure(traces, layout=go.Layout(
    title="Boolean Searches → Science Backbone (edge width ≈ sub-topic overlap)",
    showlegend=True, hovermode="closest", height=650,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor="white"))
fig_network.write_html(str(OUTPUT_DIR / "network_boolean_backbone.html"), include_plotlyjs="cdn")
fig_network.show()
print(f"✓ Saved: {OUTPUT_DIR / 'network_boolean_backbone.html'}")


✓ Saved: c:\Users\ecm3479\OneDrive - The University of Texas at Austin\Documents\GitHub\mediascrubbing-floodhealth\IntegratedResults\network_boolean_backbone.html


## 6. Burst analysis — how perceptions shift as events progress

**What this adds that prior notebooks lacked**: Kleinberg's two-state burst model identifies time windows where a term's frequency is anomalously high relative to its baseline rate. Run per event and aligned to day-zero (Helene 2024-09-26, LA Fires 2025-01-07, TX Flood 2025-07-04), this reveals which terms burst during impact vs. immediate aftermath vs. first-week retrospective vs. one-month anniversary — i.e., how the *framing* of an event shifts through its arc.

Two outputs:
- **Burst heatmap** per event: terms × days, colored by burst intensity.
- **Aligned-timeline streamgraph**: boolean-search volume vs. days-since-day-zero, overlaid across all three events for direct comparison.


### 6.1 Kleinberg two-state burst detection

Classic Kleinberg (2003) algorithm: observes inter-arrival times of a term's occurrences; finds the Viterbi path through a two-state HMM (low-rate "normal" vs. high-rate "burst"); returns burst windows with start/end and intensity. The implementation below is compact and dependency-free.


In [14]:
def kleinberg_bursts(timestamps, s=2.0, gamma=1.0):
    """
    Minimal two-state Kleinberg burst detector.
    timestamps: sorted list of datetimes at which the term occurred.
    s: state transition cost multiplier (higher = fewer, larger bursts).
    gamma: state-change penalty.
    Returns list of (start_dt, end_dt, intensity) for detected burst intervals.
    """
    if len(timestamps) < 4:
        return []
    times = sorted(timestamps)
    gaps = [(times[i+1] - times[i]).total_seconds() / 86400.0
            for i in range(len(times)-1)]
    gaps = [max(g, 1e-6) for g in gaps]
    mean_gap = np.mean(gaps)
    if mean_gap <= 0:
        return []
    # Two rates: state 0 = baseline (1/mean_gap), state 1 = burst (s/mean_gap)
    rates = [1.0 / mean_gap, s / mean_gap]
    # Viterbi over gaps
    n = len(gaps)
    cost = [[0.0, 0.0] for _ in range(n)]
    back = [[0, 0] for _ in range(n)]
    for k in (0, 1):
        cost[0][k] = -np.log(rates[k] * np.exp(-rates[k] * gaps[0]))
    for i in range(1, n):
        for k in (0, 1):
            best, best_prev = None, 0
            for j in (0, 1):
                # Kleinberg (2003): cost to transition UP into a higher state is
                # (new_state - old_state) * gamma * log(n); going DOWN is free.
                trans = (k - j) * gamma * np.log(n) if k > j else 0.0
                c = cost[i-1][j] + trans - np.log(rates[k] * np.exp(-rates[k] * gaps[i]))
                if best is None or c < best:
                    best, best_prev = c, j
            cost[i][k] = best
            back[i][k] = best_prev
    # Back-trace
    path = [0] * n
    path[-1] = 0 if cost[-1][0] <= cost[-1][1] else 1
    for i in range(n-2, -1, -1):
        path[i] = back[i+1][path[i+1]]
    # Extract burst intervals
    bursts = []
    in_burst = False
    start_idx = 0
    for i, st in enumerate(path):
        if st == 1 and not in_burst:
            in_burst, start_idx = True, i
        elif st == 0 and in_burst:
            intensity = sum(1/gaps[j] for j in range(start_idx, i)) / max(i-start_idx, 1)
            bursts.append((times[start_idx], times[i], intensity))
            in_burst = False
    if in_burst:
        intensity = sum(1/gaps[j] for j in range(start_idx, n)) / max(n-start_idx, 1)
        bursts.append((times[start_idx], times[-1], intensity))
    return bursts

print("✓ Kleinberg burst detector defined")


✓ Kleinberg burst detector defined


### 6.2 Per-event burst heatmaps

For each event, we pick the ~12 most distinctive terms across that event's boolean sub-corpora (by TF-IDF), run Kleinberg on each term's timestamp series restricted to the ±`WINDOW_DAYS` window around day-zero, and render a heatmap of days-since-day-zero × term with intensity encoding. A vertical line marks day-zero.


In [15]:
BURST_TERMS_PER_EVENT = 12

def term_timestamps(docs, term):
    mask = docs["text_content"].str.contains(rf"\b{re.escape(term)}\b", case=False, regex=True)
    return docs.loc[mask, "timestamp"].dropna().tolist()

def distinctive_terms(sub_df, all_df, n=BURST_TERMS_PER_EVENT):
    """Terms frequent in sub_df but not in (all_df minus sub_df)."""
    in_corpus  = " ".join(sub_df["text_content"].tolist())
    out_corpus = " ".join(all_df.loc[~all_df.index.isin(sub_df.index), "text_content"].tolist())
    vec = TfidfVectorizer(max_features=3000, stop_words=list(CONTENT_STOPWORDS), min_df=2)
    try:
        X = vec.fit_transform([in_corpus, out_corpus])
    except ValueError:
        return []
    diff = (X.toarray()[0] - X.toarray()[1])
    vocab = np.array(vec.get_feature_names_out())
    return list(vocab[diff.argsort()[::-1][:n]])

event_burst_figs = {}
for event, day0_str in EVENT_DAY_ZERO.items():
    day0 = datetime.fromisoformat(day0_str).replace(tzinfo=timezone.utc)
    win_start = day0 - timedelta(days=WINDOW_DAYS)
    win_end   = day0 + timedelta(days=WINDOW_DAYS)
    sub = df[df["events"].apply(lambda es: event in es) &
             df["timestamp"].between(win_start, win_end)].copy()
    if len(sub) < 10:
        print(f"  ⚠ {event}: only {len(sub)} docs in window — skipping burst heatmap")
        continue
    terms = distinctive_terms(sub, df)
    if not terms:
        continue
    # Build days-since-day-zero × term intensity matrix
    days = list(range(-WINDOW_DAYS, WINDOW_DAYS+1))
    matrix = np.zeros((len(terms), len(days)))
    for ti, term in enumerate(terms):
        ts = term_timestamps(sub, term)
        bursts = kleinberg_bursts(ts)
        for b_start, b_end, intensity in bursts:
            d0 = (b_start - day0).days
            d1 = (b_end - day0).days
            for d in range(max(d0, -WINDOW_DAYS), min(d1, WINDOW_DAYS)+1):
                matrix[ti, d - days[0]] = max(matrix[ti, d - days[0]], intensity)
    fig = go.Figure(go.Heatmap(z=matrix, x=days, y=terms, colorscale="Oranges",
                               colorbar=dict(title="burst<br>intensity")))
    fig.add_vline(x=0, line=dict(color="red", width=2, dash="dash"),
                  annotation_text=f"Day 0 ({day0_str})", annotation_position="top")
    fig.update_layout(title=f"{event} — term bursts around day-zero (±{WINDOW_DAYS} days)",
                      xaxis_title="Days since day-zero", yaxis_title="Term",
                      height=450, plot_bgcolor="white")
    fig.write_html(str(OUTPUT_DIR / f"burst_{event}.html"), include_plotlyjs="cdn")
    event_burst_figs[event] = fig
    fig.show()
    print(f"✓ {event}: {len(sub):,} docs, {len(terms)} distinctive terms")


c:\Users\ecm3479\AppData\Local\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning:

Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['001', '0121', '02', '0232', '0_tim', '10', '1022', '1059', '1075', '10778', '1078', '108', '1082', '1094', '11', '110', '1105', '1111', '1124', '1127', '1143', '117', '1172', '1181', '1188', '119', '120', '1215', '123', '1234', '1250', '1269', '1270', '13', '131', '1311', '1312', '1313', '1330', '1332', '134', '1349', '1354', '136', '1367', '137', '1385', '1391', '13ender', '1432', '14331', '1466', '1470', '1479', '148', '1485', '1498', '1502', '1509', '151', '1519', '1526', '1540', '155', '156', '1564', '1568', '1578', '1592', '1594', '1597', '1615', '1621', '1622', '1624', '1642', '1653', '1670', '1678', '168', '17', '171', '1726', '1728', '1758', '1799', '1821', '1859', '1869', '187', '1875', '1899', '19', '1908', '191', '1912', '1933', '1934', '1945', '1980', '19890'

✓ NC_Helene: 16,924 docs, 12 distinctive terms


✓ TX_Flood: 1,410 docs, 12 distinctive terms


## 7. Cross-filter comparison visualizations

This section is dedicated to *highlighting distinctions* between the boolean sub-corpora — the explicit goal you called out. Three views:

1. **Match counts per boolean search** — simple bar chart, sanity check.
2. **Aligned-timeline streamgraph** — daily volume per boolean search, x-axis is days-since-day-zero so all three events align on the same axis for comparison.
3. **Jaccard similarity matrix** — pairwise overlap between boolean sub-corpora's top-term sets; shows which boolean searches converge on shared vocabulary versus diverge.


### 7.1 Match counts and event breakdown


In [16]:
counts_df = match_summary.copy().sort_values("matches", ascending=True)
fig_counts = px.bar(counts_df, x="matches", y="boolean_search", color="event",
                    orientation="h", color_discrete_map=EVENT_COLORS,
                    title="Documents matched per boolean search")
fig_counts.update_layout(height=400, plot_bgcolor="white", yaxis_title="", xaxis_title="Matched documents")
fig_counts.write_html(str(OUTPUT_DIR / "match_counts.html"), include_plotlyjs="cdn")
fig_counts.show()


### 7.2 Aligned-timeline streamgraph — event arcs on a common axis

Daily document volume per boolean search, with each event's x-axis re-centered so day-zero = 0. Lets you compare "how does the arc of LA Fires discourse compare to NC Helene" without the events' actual calendar positions getting in the way.


In [17]:
rows = []
for event, day0_str in EVENT_DAY_ZERO.items():
    day0 = datetime.fromisoformat(day0_str).replace(tzinfo=timezone.utc)
    for bs in BOOLEAN_SEARCHES:
        if bs["event"] != event:
            continue
        sub = df[df[f"bool_{bs['name']}"] & df["timestamp"].between(
            day0 - timedelta(days=WINDOW_DAYS), day0 + timedelta(days=WINDOW_DAYS))]
        if sub.empty:
            continue
        daily = sub.groupby(sub["timestamp"].dt.date).size().reset_index(name="count")
        daily["days_since_day0"] = daily["timestamp"].apply(
            lambda d: (datetime.combine(d, datetime.min.time(), tzinfo=timezone.utc) - day0).days)
        daily["event"] = event
        daily["boolean_search"] = bs["name"]
        rows.append(daily[["days_since_day0", "count", "event", "boolean_search"]])

if rows:
    aligned = pd.concat(rows, ignore_index=True)
    fig_aligned = px.area(aligned, x="days_since_day0", y="count", color="boolean_search",
                          facet_row="event", title="Aligned timelines: daily volume vs. days-since-day-zero",
                          height=650)
    fig_aligned.add_vline(x=0, line=dict(color="red", width=1, dash="dash"))
    fig_aligned.update_layout(plot_bgcolor="white")
    fig_aligned.write_html(str(OUTPUT_DIR / "aligned_timelines.html"), include_plotlyjs="cdn")
    fig_aligned.show()
else:
    print("  No data in event windows to plot aligned timelines.")


### 7.3 Jaccard similarity between boolean-search top-term sets

For every pair of boolean searches, computes the Jaccard similarity of their top-30 TF-IDF terms. Values near 1 mean the two sub-corpora talk about the same things; values near 0 mean genuinely distinct discourse. Useful for validating that (for example) `LAfires_Misinformation` and `HELENE_Misinformation` share misinformation vocabulary but differ on event-specific terms.


In [18]:
def top_terms_set(boolean_name, n=30):
    sub = df[df[f"bool_{boolean_name}"]]
    if len(sub) < 3:
        return set()
    return set(top_terms_tfidf(sub["text_content"].tolist(), CONTENT_STOPWORDS, n=n))

names = [bs["name"] for bs in BOOLEAN_SEARCHES]
term_sets = {n: top_terms_set(n) for n in names}
J = np.zeros((len(names), len(names)))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        u = term_sets[a] | term_sets[b]
        J[i, j] = len(term_sets[a] & term_sets[b]) / len(u) if u else 0.0

fig_jac = go.Figure(go.Heatmap(z=J, x=names, y=names, colorscale="Blues",
                               zmin=0, zmax=1, colorbar=dict(title="Jaccard")))
fig_jac.update_layout(title="Boolean-search vocabulary overlap (top-30 TF-IDF terms)",
                      height=550, xaxis_tickangle=-30, plot_bgcolor="white")
fig_jac.write_html(str(OUTPUT_DIR / "jaccard_similarity.html"), include_plotlyjs="cdn")
fig_jac.show()


c:\Users\ecm3479\AppData\Local\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning:

Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['001', '0121', '02', '0232', '0_tim', '10', '1022', '1059', '1075', '10778', '1078', '108', '1082', '1094', '11', '110', '1105', '1111', '1124', '1127', '1143', '117', '1172', '1181', '1188', '119', '120', '1215', '123', '1234', '1250', '1269', '1270', '13', '131', '1311', '1312', '1313', '1330', '1332', '134', '1349', '1354', '136', '1367', '137', '1385', '1391', '13ender', '1432', '14331', '1466', '1470', '1479', '148', '1485', '1498', '1502', '1509', '151', '1519', '1526', '1540', '155', '156', '1564', '1568', '1578', '1592', '1594', '1597', '1615', '1621', '1622', '1624', '1642', '1653', '1670', '1678', '168', '17', '171', '1726', '1728', '1758', '1799', '1821', '1859', '1869', '187', '1875', '1899', '19', '1908', '191', '1912', '1933', '1934', '1945', '1980', '19890'

## 8. Export

Writes everything to `OUTPUT_DIR`:
- `matched_documents.csv` — every matched row, all boolean tags, ready for qualitative coding
- `subtopics.csv` — discovered sub-topics per boolean search, with backbone mappings
- `match_summary.csv` — counts per boolean search
- Plus the five interactive HTML figures already saved in Sections 5–7
- `report.md` — one-page Markdown summary


In [24]:
import subprocess
import sys

# Install tabulate in the current kernel's environment
subprocess.check_call([sys.executable, "-m", "pip", "install", "tabulate"])

print("✅ tabulate installed successfully! Re-run your original cell now.")

✅ tabulate installed successfully! Re-run your original cell now.


In [25]:
from datetime import datetime as _dt
stamp = _dt.now().strftime("%Y%m%d_%H%M%S")

# Flatten list-valued columns for CSV
export_df = df.copy()
export_df["boolean_matches"] = export_df["boolean_matches"].str.join("|")
export_df["events"] = export_df["events"].str.join("|")
export_df.to_csv(OUTPUT_DIR / f"matched_documents_{stamp}.csv", index=False, encoding="utf-8-sig")

st_export = subtopics_df.copy()
st_export["top_terms"] = st_export["top_terms"].str.join("|")
st_export["backbone_mapping"] = st_export["backbone_mapping"].apply(
    lambda hits: "|".join(f"{d}>{s}({sc})" for d, s, sc in hits))
st_export.to_csv(OUTPUT_DIR / f"subtopics_{stamp}.csv", index=False, encoding="utf-8-sig")
match_summary.to_csv(OUTPUT_DIR / f"match_summary_{stamp}.csv", index=False)

# Short Markdown report
report = [f"# EPH × Semantic Bridge × Burst — Run {stamp}",
          f"- Data source: `{DATA_DIR}`",
          f"- Total documents ingested: {len(df_raw):,}",
          f"- Documents matching at least one boolean search: {len(df):,}",
          f"- Boolean searches evaluated: {len(BOOLEAN_SEARCHES)}",
          f"- Sub-topics discovered: {len(subtopics_df)}",
          "",
          "## Match counts", match_summary.to_markdown(index=False),
          "",
          "## Sub-topics", st_export[['boolean_search','subtopic_id','primary_domain','top_terms']].to_markdown(index=False),
          "",
          "## Outputs",
          f"- `{OUTPUT_DIR / f'matched_documents_{stamp}.csv'}`",
          f"- `{OUTPUT_DIR / f'subtopics_{stamp}.csv'}`",
          f"- `{OUTPUT_DIR / 'network_boolean_backbone.html'}`",
          f"- `{OUTPUT_DIR / 'match_counts.html'}`",
          f"- `{OUTPUT_DIR / 'aligned_timelines.html'}`",
          f"- `{OUTPUT_DIR / 'jaccard_similarity.html'}`"]
for evt in EVENT_DAY_ZERO:
    fp = OUTPUT_DIR / f"burst_{evt}.html"
    if fp.exists():
        report.append(f"- `{fp}`")
(OUTPUT_DIR / f"report_{stamp}.md").write_text("\n".join(report))
print(f"✓ Exports written to {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.glob(f"*{stamp}*")):
    print(f"  - {f.name}")


✓ Exports written to C:\Users\ecm3479\OneDrive - The University of Texas at Austin\Documents\Media Scrubbing Worksheets\IntegratedResults
  - match_summary_20260605_151746.csv
  - matched_documents_20260605_151746.csv
  - report_20260605_151746.md
  - subtopics_20260605_151746.csv


---

## What this notebook does, at a glance

| Section | Role | Source |
|---|---|---|
| 1 | Setup, paths, imports | new |
| 2 | JSON/JSONL ingest, content vs. metadata split | carried from EPH, restructured |
| 3 | 11 keyword groups → 7 boolean searches → per-doc tagging | carried from EPH |
| 4 | Sub-topics *inside* each boolean corpus; optional LLM labels | new (replaces whole-corpus LDA) |
| 5 | Boolean searches ↔ ETA science backbone network | adapted from Load & Bridge |
| 6 | Kleinberg burst detection per event, aligned to day-zero | new |
| 7 | Cross-filter comparison: counts, aligned streamgraph, Jaccard | new |
| 8 | CSV + HTML + Markdown exports | new |

## Things you may want to edit

- **`DATA_DIR`** (§1.1) — point at your Corral path or local folder
- **`_GROUP_REGISTRY` / `BOOLEAN_SEARCHES`** (§3.1) — add/remove keyword groups and searches
- **`EVENT_DAY_ZERO` / `WINDOW_DAYS`** (§1.1) — change event anchors or analysis window
- **`ENABLE_LLM_LABELS`** (§4.3) — turn on LLM topic labeling once you set `ANTHROPIC_API_KEY`
- **`ETO_EXPORT_PATH`** (§5.1) — supply a real ETO Map-of-Science export to replace the default backbone
